# Explosão de BOM multinível — Google Colab

Este notebook solicita **apenas dois arquivos Excel**, em **duas caixas de seleção separadas**:

1. **LISTAGEM** — selecione primeiro este arquivo.
2. **RECEITAS** — uma segunda caixa será aberta logo depois para selecionar este arquivo.

Não é necessário enviar o arquivo GRU JJ INT 0262: o código já mantém os nomes e a ordem das suas **17 colunas**.

O processamento conserva a estrutura hierárquica atual (pai → receitas filhas → componentes em sequência), mantém a aba `Pendencias`, aplica formato numérico nas colunas `Yield`, `Quantidade` e `Custo Unitario` e inicia o download automático com o nome **`MRP Costing Teknisa + [CIA].xlsx`**, usando o nome da companhia identificado na LISTAGEM.

Execute a célula de código abaixo. Selecione somente um arquivo em cada caixa de diálogo.

In [ ]:
# -*- coding: utf-8 -*-
"""
BOM / Estrutura de Produto multinível para relatórios Excel.

Como usar no Google Colab:
1) Abra/importa este código em uma célula do Google Colab ou use o notebook .ipynb.
2) Execute a célula.
3) Selecione primeiro o arquivo LISTAGEM e, em uma segunda caixa separada, o arquivo RECEITAS.
4) O Excel resultante será salvo como "MRP Costing Teknisa + [CIA].xlsx" e baixado automaticamente.

O modelo de saída (17 colunas e seus nomes) está definido no próprio script, portanto
não é necessário selecionar o GRU JJ INT 0262 in cada execução.
Bibliotecas: pandas e openpyxl (além do módulo google.colab no Colab).
"""

from __future__ import annotations

import math
import os
import re
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter


# -----------------------------------------------------------------------------
# 1) CONFIGURAÇÃO DAS COLUNAS DOS RELATÓRIOS
# -----------------------------------------------------------------------------
# Os arquivos enviados foram lidos como relatórios sem cabeçalho tabular padrão.
# LISTAGEM (índices abaixo são base 0: A=0, B=1, ...):
COL_LIST_CODIGO = 0       # Coluna A: Produto (código do produto/receita)
COL_LIST_DESCRICAO = 1    # Coluna B: Descrição
COL_LIST_PRECO_BASE = 2   # Coluna C: P.Base / custo-base unitário
COL_LIST_APROVACAO = 3    # Coluna D: Aprov (rendimento/aproveitamento, ex.: 0.98)
COL_LIST_QUANTIDADE = 5   # Coluna F: Qtde
COL_LIST_UNIDADE = 6      # Coluna G: Un
COL_LIST_PERCENTUAL = 7   # Coluna H: %
COL_LIST_CUSTO = 8         # Coluna I: Custo

# RECEITAS:
# A contém "CÓDIGO    DESCRIÇÃO"; F=Estoque/preço-base; G=Aprov; I=Quant.; J=Un.
COL_REC_ITEM = 0          # Coluna A: código e descrição juntos
COL_REC_PRECO_BASE = 5    # Coluna F: Estoque (preço-base unitário)
COL_REC_APROVACAO = 6     # Coluna G: Aprov.
COL_REC_QUANTIDADE = 8    # Coluna I: Quant.
COL_REC_UNIDADE = 9       # Coluna J: Un.

# Modelo GRU JJ INT 0262:
# Esta lista reproduz os nomes e a ordem das 17 colunas do modelo usado como referência.
# Assim, no Colab, o usuário precisa selecionar SOMENTE LISTAGEM e RECEITAS.
COLUNAS_MODELO = [
    "Base", "Cod Menu Grid", "Base de custo", "Cia", "Serviço", "Cardapio",
    "Relatorio", "Class", "Codigo inflightor", "Produto e receita", "Classe",
    "Ciclo", "%", "Yield", "Quantidade", "Unidade", "Custo Unitario",
]


# -----------------------------------------------------------------------------
# 2) FUNÇÕES DE LIMPEZA E CONVERSÃO
# -----------------------------------------------------------------------------
def vazio(valor: Any) -> bool:
    """True quando a célula está vazia/NaN."""
    if valor is None:
        return True
    try:
        return bool(pd.isna(valor))
    except (TypeError, ValueError):
        return False


def texto(valor: Any) -> str:
    if vazio(valor):
        return ""
    return re.sub(r"\s+", " ", str(valor)).strip()


def sem_acentos(valor: Any) -> str:
    s = unicodedata.normalize("NFKD", texto(valor))
    return "".join(ch for ch in s if not unicodedata.combining(ch)).upper().strip()


def numero(valor: Any) -> Optional[float]:
    """Converte número Excel/texto para float; aceita decimal com vírgula."""
    if vazio(valor):
        return None
    if isinstance(valor, bool):
        return float(valor)
    if isinstance(valor, (int, float)):
        n = float(valor)
        return n if math.isfinite(n) else None
    s = texto(valor).replace(" ", "")
    if not s or s in {"-", "—", "–"}:
        return None
    # Se houver ponto e vírgula, presume-se que o último separador é decimal.
    if "," in s and "." in s:
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        s = s.replace(",", ".")
    try:
        n = float(s)
        return n if math.isfinite(n) else None
    except (TypeError, ValueError):
        return None


def codigo_descricao(valor: Any) -> Tuple[str, str]:
    """Separa o primeiro token (código) da descrição na coluna A de RECEITAS."""
    s = texto(valor)
    if not s:
        return "", ""
    partes = s.split(maxsplit=1)
    return partes[0].strip(), (partes[1].strip() if len(partes) > 1 else "")


def limpar_codigo_servico(nome_servico: Any) -> Optional[str]:
    """Remove o código numérico/alfanumérico inicial que antecede o nome do serviço (ex: '01010101010101017 SERVICO' -> 'SERVICO')."""
    s = texto(nome_servico)
    if not s:
        return None
    # Padrão: busca sequências de dígitos, letras, hífens ou pontos com tamanho de 10 a 20 caracteres no início, seguidos de espaço
    padrao_codigo = re.compile(r"^[A-Z0-9.\-]{10,22}\s+(.+)$", re.IGNORECASE)
    m = padrao_codigo.match(s)
    if m:
        return m.group(1).strip()
    return s


def extrair_apenas_data(valor_custo: Any) -> Optional[str]:
    """Extrai apenas a primeira ocorrência de data (DD/MM/AAAA ou similar) e remove o que vem depois."""
    s = texto(valor_custo)
    if not s:
        return None
    # Busca padrões comuns de data como DD/MM/AAAA ou DD/MM/AA ou YYYY-MM-DD
    padrao_data = re.compile(r"(\b\d{1,2}[/\-.]\d{1,2}[/\-.]\d{2,4}\b|\b\d{4}[/\-.]\d{1,2}[/\-.]\d{1,2}\b)")
    m = padrao_data.search(s)
    if m:
        return m.group(1)
    return s


def rendimento_valido(valor: Any) -> float:
    """Aprovação válida para cálculo. Valores ausentes/atípicos viram 1.0."""
    n = numero(valor)
    # Nos relatórios RECEITAS, 0 ou 0.0001 frequentemente é um marcador em
    # linhas que são receitas intermediárias, e não um rendimento de matéria-prima.
    if n is None or n <= 0 or n > 1:
        return 1.0
    if n < 0.001:
        return 1.0
    return n


def percentual_texto(valor: Any) -> Any:
    n = numero(valor)
    if n is None:
        s = texto(valor)
        return s if s else None
    # A LISTAGEM enviada representa percentuais como 100, 40 etc.; alguns
    # relatórios podem trazer 1.0 para representar 100%.
    p = n * 100 if 0 <= n <= 1 else n
    if abs(p - round(p)) < 1e-8:
        return f"{int(round(p))}%"
    return f"{p:g}%"


def primeira_celula_a_direita(df: pd.DataFrame, rotulo: str, max_direita: int = 6) -> Any:
    """Busca um rótulo em qualquer célula e retorna a próxima célula preenchida à direita."""
    rotulo_n = sem_acentos(rotulo)
    for r in range(len(df)):
        for c in range(df.shape[1]):
            if sem_acentos(df.iat[r, c]) == rotulo_n:
                limite = min(df.shape[1], c + max_direita + 1)
                for cc in range(c + 1, limite):
                    if not vazio(df.iat[r, cc]):
                        return df.iat[r, cc]
    return None


def encontrar_metadados_gerais(df_listagem: pd.DataFrame) -> Dict[str, Any]:
    """Encontra metadados de escopo geral do arquivo LISTAGEM."""
    filial = primeira_celula_a_direita(df_listagem, "Filial")
    manual = primeira_celula_a_direita(df_listagem, "Manual")
    custo = primeira_celula_a_direita(df_listagem, "Custo")
    cia = primeira_celula_a_direita(df_listagem, "Companhia")

    manual_txt = texto(manual)
    cod_menu = manual_txt.split(maxsplit=1)[0] if manual_txt else ""

    # Limpa o texto da data em custo
    data_custo = extrair_apenas_data(custo)

    return {
        "Base": texto(filial) or None,
        "Cod Menu Grid": cod_menu or None,
        "Base de custo": data_custo or None,
        "Cia": texto(cia) or None,
    }


# -----------------------------------------------------------------------------
# 3) VERTICALIZAÇÃO DOS RELATÓRIOS
# -----------------------------------------------------------------------------
def ler_listagem(df: pd.DataFrame) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Transforma a LISTAGEM em relações Prato/Grupo -> Item e rastreia Classe/Ciclo/Serviço dinamicamente."""
    linhas: List[Dict[str, Any]] = []
    relacoes: List[Dict[str, Any]] = []

    grupo_atual = ""
    classe_atual = None
    ciclo_atual = None
    servico_atual = None

    for idx in range(len(df)):
        row = df.iloc[idx]

        # Rastreia metadados que mudam dinamicamente ao longo das linhas do relatório
        for col_idx in range(df.shape[1]):
            val = row.iloc[col_idx]
            if vazio(val):
                continue
            val_normalizado = sem_acentos(val)

            if val_normalizado == "CLASSE":
                # Procura a classe preenchida à direita nesta mesma linha
                for cc in range(col_idx + 1, df.shape[1]):
                    if not vazio(row.iloc[cc]):
                        classe_atual = texto(row.iloc[cc])
                        break
            elif val_normalizado == "CICLO":
                # Procura o ciclo preenchido à direita nesta mesma linha
                for cc in range(col_idx + 1, df.shape[1]):
                    if not vazio(row.iloc[cc]):
                        ciclo_val = row.iloc[cc]
                        ciclo_atual = numero(ciclo_val) if numero(ciclo_val) is not None else texto(ciclo_val) or None
                        break
            elif val_normalizado == "CARDAPIO":
                # Procura o serviço preenchido à direita (coluna B)
                for cc in range(col_idx + 1, df.shape[1]):
                    if not vazio(row.iloc[cc]):
                        servico_atual = limpar_codigo_servico(row.iloc[cc])
                        break

        a = row.iloc[COL_LIST_CODIGO] if df.shape[1] > COL_LIST_CODIGO else None
        b = row.iloc[COL_LIST_DESCRICAO] if df.shape[1] > COL_LIST_DESCRICAO else None
        c = row.iloc[COL_LIST_PRECO_BASE] if df.shape[1] > COL_LIST_PRECO_BASE else None
        q = row.iloc[COL_LIST_QUANTIDADE] if df.shape[1] > COL_LIST_QUANTIDADE else None
        un = row.iloc[COL_LIST_UNIDADE] if df.shape[1] > COL_LIST_UNIDADE else None

        # Cabeçalho de prato/grupo: conteúdo apenas na coluna B, sem código na A.
        restantes_vazias = all(vazio(row.iloc[j]) for j in range(2, df.shape[1]))
        if not vazio(b) and vazio(a) and restantes_vazias:
            candidato = texto(b)
            # Evita interpretar cabeçalhos gerais repetidos como pratos.
            if sem_acentos(candidato) not in {"DESCRICAO", "TOTAL", "TOTAL DO MENU"}:
                grupo_atual = candidato
            continue

        # Linha de componente: código + descrição + quantidade/unidade.
        qn = numero(q)
        if vazio(a) or vazio(b) or qn is None or vazio(un):
            continue
        codigo = texto(a)
        descricao = texto(b)
        if sem_acentos(codigo) in {"PRODUTO", "TOTAL", "TOTAL DO MENU"}:
            continue
        if not grupo_atual:
            continue

        item = {
            "grupo": grupo_atual,
            "codigo": codigo,
            "descricao": descricao,
            "preco_base": numero(c),
            "aprovacao": numero(row.iloc[COL_LIST_APROVACAO]) if df.shape[1] > COL_LIST_APROVACAO else None,
            "quantidade": qn,
            "unidade": texto(un),
            "percentual": row.iloc[COL_LIST_PERCENTUAL] if df.shape[1] > COL_LIST_PERCENTUAL else None,
            "custo_origem": numero(row.iloc[COL_LIST_CUSTO]) if df.shape[1] > COL_LIST_CUSTO else None,
            "linha_origem": idx + 1,
            "classe_dinamica": classe_atual,
            "ciclo_dinamico": ciclo_atual,
            "servico_dinamico": servico_atual,
        }
        linhas.append(item)
        relacoes.append({
            "Origem": "LISTAGEM",
            "Código do pai": grupo_atual,
            "Pai": grupo_atual,
            "Código do filho": codigo,
            "Filho": descricao,
            "Quantidade informada": qn,
            "Unidade": texto(un),
            "Rendimento do pai": None,
            "Rendimento do filho": item["aprovacao"],
            "Preço-base do filho": item["preco_base"],
            "Linha no arquivo": idx + 1,
        })

    return linhas, relacoes


def ler_receitas(df: pd.DataFrame) -> Tuple[Dict[str, Dict[str, Any]], List[Dict[str, Any]], List[Dict[str, Any]]]:
    """Lê blocos de receita e transforma os componentes em relações PAI -> FILHO."""
    receitas: Dict[str, Dict[str, Any]] = {}
    relacoes: List[Dict[str, Any]] = []
    pendencias: List[Dict[str, Any]] = []
    atual: Optional[Dict[str, Any]] = None

    for idx in range(len(df)):
        row = df.iloc[idx]
        raw = row.iloc[COL_REC_ITEM] if df.shape[1] > COL_REC_ITEM else None
        raw_txt = texto(raw)
        if not raw_txt:
            continue

        if sem_acentos(raw_txt) == "RENDIMENTO":
            if atual is not None:
                rendimento = numero(row.iloc[1]) if df.shape[1] > 1 else None
                unidade_rend = texto(row.iloc[2]) if df.shape[1] > 2 else ""
                atual["rendimentos"].append({
                    "quantidade": rendimento,
                    "unidade": unidade_rend,
                    "linha": idx + 1,
                })
            continue

        # Cabeçalho de receita: código R.* na coluna A e demais colunas vazias.
        outras_vazias = all(vazio(row.iloc[j]) for j in range(1, df.shape[1]))
        if raw_txt.upper().startswith("R.") and outras_vazias:
            cod, desc = codigo_descricao(raw_txt)
            atual = {
                "codigo": cod,
                "descricao": desc,
                "componentes": [],
                "rendimentos": [],
                "linha": idx + 1,
            }
            receitas[cod] = atual
            continue

        if atual is None:
            continue

        quantidade = numero(row.iloc[COL_REC_QUANTIDADE]) if df.shape[1] > COL_REC_QUANTIDADE else None
        unidade = texto(row.iloc[COL_REC_UNIDADE]) if df.shape[1] > COL_REC_UNIDADE else ""
        if quantidade is None or not unidade:
            continue

        codigo, descricao = codigo_descricao(raw_txt)
        if not codigo:
            continue
        comp = {
            "codigo": codigo,
            "descricao": descricao or raw_txt,
            "quantidade": quantidade,
            "unidade": unidade,
            "aprovacao": numero(row.iloc[COL_REC_APROVACAO]) if df.shape[1] > COL_REC_APROVACAO else None,
            "preco_base": numero(row.iloc[COL_REC_PRECO_BASE]) if df.shape[1] > COL_REC_PRECO_BASE else None,
            "linha_origem": idx + 1,
            "pai_codigo": atual["codigo"],
            "pai_descricao": atual["descricao"],
        }
        atual["componentes"].append(comp)
        relacoes.append({
            "Origem": "RECEITAS",
            "Código do pai": atual["codigo"],
            "Pai": atual["descricao"],
            "Código do filho": codigo,
            "Filho": comp["descricao"],
            "Quantidade informada": quantidade,
            "Unidade": unidade,
            "Rendimento do pai": None,
            "Rendimento do filho": comp["aprovacao"],
            "Preço-base do filho": comp["preco_base"],
            "Linha no arquivo": idx + 1,
        })

    # O primeiro "Rendimento" é o rendimento produzido do lote; frequentemente
    # o segundo é a referência de custo para 1 unidade (1 KG, por exemplo).
    for codigo, receita in receitas.items():
        rendimentos = receita["rendimentos"]
        principal = next((r for r in rendimentos if r["quantidade"] is not None and r["quantidade"] > 0), None)
        if principal is None:
            receita["rendimento"] = 1.0
            receita["unidade_rendimento"] = ""
            pendencias.append({
                "Tipo": "Rendimento ausente",
                "Código": codigo,
                "Descrição": receita["descricao"],
                "Contexto": f"Linha {receita['linha']} de RECEITAS",
                "Detalhe": "A receita não tem rendimento positivo legível; foi usado 1.0 como padrão.",
            })
        else:
            receita["rendimento"] = float(principal["quantidade"])
            receita["unidade_rendimento"] = principal["unidade"]

    # Atualiza rendimento do pai na tabela verticalizada.
    mapa_rendimento = {c: r.get("rendimento", 1.0) for c, r in receitas.items()}
    for rel in relacoes:
        if rel["Origem"] == "RECEITAS":
            rel["Rendimento do pai"] = mapa_rendimento.get(rel["Código do pai"], 1.0)

    return receitas, relacoes, pendencias


# -----------------------------------------------------------------------------
# 4) EXPLOSÃO RECURSIVA ATÉ OS INGREDIENTES FINAIS
# -----------------------------------------------------------------------------
def processar_arquivos(
    caminho_listagem: str,
    caminho_receitas: str,
    caminho_saida: Optional[str] = None,
) -> str:
    """Gera a BOM hierárquica usando apenas LISTAGEM e RECEITAS."""
    # header=None é importante: os relatórios têm títulos e metadados antes da tabela.
    df_list = pd.read_excel(caminho_listagem, sheet_name=0, header=None, dtype=object, engine="openpyxl")
    df_rec = pd.read_excel(caminho_receitas, sheet_name=0, header=None, dtype=object, engine="openpyxl")

    # As 17 colunas seguem os nomes e a ordem do modelo GRU JJ INT 0262.
    colunas_modelo = COLUNAS_MODELO.copy()

    # Encontra os metadados fixos
    metadados_gerais = encontrar_metadados_gerais(df_list)

    if not caminho_saida:
        # O nome de saída usa a companhia encontrada na própria LISTAGEM.
        cia_nome = texto(metadados_gerais.get("Cia")) or "CIA_NAO_IDENTIFICADA"
        # Remove apenas caracteres inválidos em nomes de arquivos; mantém espaços e acentos.
        cia_nome = re.sub(r'[<>:"/\\|?*\x00-\x1f]', " ", cia_nome)
        cia_nome = re.sub(r"\s+", " ", cia_nome).strip(" .") or "CIA_NAO_IDENTIFICADA"
        caminho_saida = f"MRP Costing Teknisa {cia_nome}.xlsx"

    itens_listagem, _ = ler_listagem(df_list)
    receitas, _, pendencias = ler_receitas(df_rec)

    # Cada linha é adicionada na ordem da árvore. O nível é usado apenas na formatação,
    # sem criar uma coluna extra que altere as 17 colunas do modelo GRU.
    linhas_estrutura: List[Dict[str, Any]] = []
    detalhes_visuais: List[Dict[str, Any]] = []
    pendencias_codigos = set()
    pendencias_sem_componentes = set()
    pendencias_ciclos = set()

    def adicionar_linha(
        grupo: str,
        codigo: str,
        descricao: str,
        qtd_liquida: Optional[float],
        unidade: str,
        aprovacao: Any,
        preco_base: Any,
        percentual: Any,
        relatorio: str,
        classe_item: Optional[str],
        nivel: int,
        contexto: str,
        classe_dinamica: Optional[str],
        ciclo_dinamico: Any,
        servico_dinamico: Optional[str],
        tipo_linha: str = "item",
        receita_nao_encontrada: bool = False,
    ) -> None:
        """Inclui uma linha de pai/receita/ingrediente no formato do modelo."""
        fator = rendimento_valido(aprovacao)
        if receita_nao_encontrada:
            fator = 1.0
        qtd_liquida_num = numero(qtd_liquida)
        qtd_bruta = (qtd_liquida_num / fator) if qtd_liquida_num is not None and fator else qtd_liquida_num
        preco = numero(preco_base)
        custo_total = qtd_bruta * preco if qtd_bruta is not None and preco is not None else None

        linha = {c: None for c in colunas_modelo}
        linha.update(metadados_gerais)
        linha.update({
            "Serviço": servico_dinamico,
            "Cardapio": grupo,
            "Relatorio": relatorio,
            # 'Class' recebe o identificador local (R, S, P)
            "Class": classe_item,
            "Codigo inflightor": codigo or None,
            "Produto e receita": descricao or None,
            # 'Classe' recebe o valor dinâmico da listagem correspondente
            "Classe": classe_dinamica,
            "Ciclo": ciclo_dinamico,
            "%": percentual_texto(percentual) if percentual is not None else None,
            "Yield": fator if tipo_linha != "pai" else None,
            "Quantidade": 1 if tipo_linha == "pai" else qtd_bruta,
            "Unidade": unidade or None,
            # Para itens com preço-base, mantém o cálculo usado na versão anterior.
            "Custo Unitario": None if tipo_linha == "pai" else custo_total,
        })
        linhas_estrutura.append(linha)
        detalhes_visuais.append({"nivel": nivel, "tipo": tipo_linha})

    def registrar_receita_nao_encontrada(codigo: str, descricao: str, contexto: str) -> None:
        if codigo not in pendencias_codigos:
            pendencias_codigos.add(codigo)
            pendencias.append({
                "Tipo": "Receita não localizada",
                "Código": codigo,
                "Descrição": descricao,
                "Contexto": contexto,
                "Detalhe": (
                    "O código parece ser uma receita (R.*), mas não há bloco correspondente em RECEITAS. "
                    "Foi mantido na estrutura, com Yield=1 e sem explosão dos componentes."
                ),
            })

    def explodir_receita(
        codigo_receita: str,
        quantidade_desejada: float,
        grupo: str,
        percentual: Any,
        caminho: List[str],
        origem_contexto: str,
        classe_dinamica: Optional[str],
        ciclo_dinamico: Any,
        servico_dinamico: Optional[str],
        unidade_desejada: str = "",
        preco_base_desejado: Any = None,
        aprovacao_desejada: Any = None,
        descricao_desejada: str = "",
        nivel: int = 1,
        relatorio: str = "List",
        classe: str = "R",
    ) -> None:
        """Emite primeiro a linha da receita e depois seus componentes recursivamente."""
        receita = receitas.get(codigo_receita)
        descricao = descricao_desejada or (receita.get("descricao", "") if receita else codigo_receita)

        # A receita sempre aparece na estrutura, mesmo que esteja sem detalhamento.
        adicionar_linha(
            grupo=grupo,
            codigo=codigo_receita,
            descricao=descricao,
            qtd_liquida=quantidade_desejada,
            unidade=unidade_desejada or (receita.get("unidade_rendimento", "") if receita else ""),
            aprovacao=aprovacao_desejada,
            preco_base=preco_base_desejado,
            percentual=percentual,
            relatorio=relatorio,
            classe_item=classe,
            nivel=nivel,
            contexto=origem_contexto,
            classe_dinamica=classe_dinamica,
            ciclo_dinamico=ciclo_dinamico,
            servico_dinamico=servico_dinamico,
            tipo_linha="receita",
            receita_nao_encontrada=receita is None,
        )

        if receita is None:
            registrar_receita_nao_encontrada(codigo_receita, descricao, origem_contexto)
            return

        if codigo_receita in caminho:
            ciclo_key = tuple(caminho + [codigo_receita])
            if ciclo_key not in pendencias_ciclos:
                pendencias_ciclos.add(ciclo_key)
                pendencias.append({
                    "Tipo": "Ciclo de receitas",
                    "Código": codigo_receita,
                    "Descrição": descricao,
                    "Contexto": origem_contexto,
                    "Detalhe": (
                        "Ciclo detectado: " + " -> ".join(caminho + [codigo_receita])
                        + ". Este ramo foi interrompido para evitar recursão infinita."
                    ),
                })
            return

        if not receita["componentes"]:
            if codigo_receita not in pendencias_sem_componentes:
                pendencias_sem_componentes.add(codigo_receita)
                pendencias.append({
                    "Tipo": "Receita sem componentes",
                    "Código": codigo_receita,
                    "Descrição": receita["descricao"],
                    "Contexto": origem_contexto,
                    "Detalhe": (
                        "A receita foi encontrada, mas não contém ingredientes com quantidade/unidade legíveis. "
                        "Ela permanece visível como nó final provisório."
                    ),
                })
            return

        rendimento = numero(receita.get("rendimento"))
        if rendimento is None or rendimento <= 0:
            rendimento = 1.0
        fator_lote = quantidade_desejada / rendimento

        for comp in receita["componentes"]:
            qtd_filho = fator_lote * comp["quantidade"]
            cod_filho = comp["codigo"]
            desc_filho = comp["descricao"]
            contexto_filho = (
                f"{grupo} > {codigo_receita} {descricao} > {cod_filho} {desc_filho}"
            )

            if cod_filho in receitas:
                # A sub-receita aparece antes dos próprios ingredientes, na linha seguinte.
                explodir_receita(
                    codigo_receita=cod_filho,
                    quantidade_desejada=qtd_filho,
                    grupo=grupo,
                    percentual=percentual,
                    caminho=caminho + [codigo_receita],
                    origem_contexto=contexto_filho,
                    classe_dinamica=classe_dinamica,
                    ciclo_dinamico=ciclo_dinamico,
                    servico_dinamico=servico_dinamico,
                    unidade_desejada=comp["unidade"],
                    preco_base_desejado=comp["preco_base"],
                    aprovacao_desejada=comp["aprovacao"],
                    descricao_desejada=desc_filho,
                    nivel=nivel + 1,
                    relatorio="Rec",
                    classe="S",
                )
            elif cod_filho.upper().startswith("R."):
                explodir_receita(
                    codigo_receita=cod_filho,
                    quantidade_desejada=qtd_filho,
                    grupo=grupo,
                    percentual=percentual,
                    caminho=caminho + [codigo_receita],
                    origem_contexto=contexto_filho,
                    classe_dinamica=classe_dinamica,
                    ciclo_dinamico=ciclo_dinamico,
                    servico_dinamico=servico_dinamico,
                    unidade_desejada=comp["unidade"],
                    preco_base_desejado=comp["preco_base"],
                    aprovacao_desejada=comp["aprovacao"],
                    descricao_desejada=desc_filho,
                    nivel=nivel + 1,
                    relatorio="Rec",
                    classe="S",
                )
            else:
                adicionar_linha(
                    grupo=grupo,
                    codigo=cod_filho,
                    descricao=desc_filho,
                    qtd_liquida=qtd_filho,
                    unidade=comp["unidade"],
                    aprovacao=comp["aprovacao"],
                    preco_base=comp["preco_base"],
                    percentual=percentual,
                    relatorio="Rec",
                    classe_item="P",
                    nivel=nivel + 1,
                    contexto=contexto_filho,
                    classe_dinamica=classe_dinamica,
                    ciclo_dinamico=ciclo_dinamico,
                    servico_dinamico=servico_dinamico,
                    tipo_linha="ingrediente",
                )

    grupo_atual_saida = None
    for item in itens_listagem:
        grupo = item["grupo"]
        classe_dinamica = item["classe_dinamica"]
        ciclo_dinamico = item["ciclo_dinamico"]
        servico_dinamico = item["servico_dinamico"]

        if grupo != grupo_atual_saida:
            # Linha pai do cardápio/prato, equivalente à primeira linha do exemplo GRU.
            adicionar_linha(
                grupo=grupo,
                codigo="",
                descricao=grupo,
                qtd_liquida=1,
                unidade="",
                aprovacao=None,
                preco_base=None,
                percentual=None,
                relatorio="List",
                classe_item=None,
                nivel=0,
                contexto=f"Prato/cardápio: {grupo}",
                classe_dinamica=classe_dinamica,
                ciclo_dinamico=ciclo_dinamico,
                servico_dinamico=servico_dinamico,
                tipo_linha="pai",
            )
            grupo_atual_saida = grupo

        contexto = (
            f"LISTAGEM linha {item['linha_origem']}: {grupo} > "
            f"{item['codigo']} {item['descricao']}"
        )
        codigo_item = item["codigo"]
        if codigo_item in receitas:
            explodir_receita(
                codigo_receita=codigo_item,
                quantidade_desejada=item["quantidade"],
                grupo=grupo,
                percentual=item["percentual"],
                caminho=[],
                origem_contexto=contexto,
                classe_dinamica=classe_dinamica,
                ciclo_dinamico=ciclo_dinamico,
                servico_dinamico=servico_dinamico,
                unidade_desejada=item["unidade"],
                preco_base_desejado=item["preco_base"],
                aprovacao_desejada=item["aprovacao"],
                descricao_desejada=item["descricao"],
                nivel=1,
                relatorio="List",
                classe="R",
            )
        elif codigo_item.upper().startswith("R."):
            # Mantém a receita não encontrada como filho imediato do prato.
            adicionar_linha(
                grupo=grupo,
                codigo=codigo_item,
                descricao=item["descricao"],
                qtd_liquida=item["quantidade"],
                unidade=item["unidade"],
                aprovacao=1.0,
                preco_base=item["preco_base"],
                percentual=item["percentual"],
                relatorio="List",
                classe_item="R",
                nivel=1,
                contexto=contexto,
                classe_dinamica=classe_dinamica,
                ciclo_dinamico=ciclo_dinamico,
                servico_dinamico=servico_dinamico,
                tipo_linha="receita",
                receita_nao_encontrada=True,
            )
            registrar_receita_nao_encontrada(codigo_item, item["descricao"], contexto)
        else:
            adicionar_linha(
                grupo=grupo,
                codigo=codigo_item,
                descricao=item["descricao"],
                qtd_liquida=item["quantidade"],
                unidade=item["unidade"],
                aprovacao=item["aprovacao"],
                preco_base=item["preco_base"],
                percentual=item["percentual"],
                relatorio="List",
                classe_item="P",
                nivel=1,
                contexto=contexto,
                classe_dinamica=classe_dinamica,
                ciclo_dinamico=ciclo_dinamico,
                servico_dinamico=servico_dinamico,
                tipo_linha="ingrediente",
            )

    df_saida = pd.DataFrame(linhas_estrutura, columns=colunas_modelo)
    df_pendencias = pd.DataFrame(pendencias, columns=[
        "Tipo", "Código", "Descrição", "Contexto", "Detalhe",
    ])

    # Uma única aba principal com as 17 colunas do modelo + uma aba de pendências.
    with pd.ExcelWriter(caminho_saida, engine="openpyxl") as writer:
        df_saida.to_excel(writer, sheet_name="Dados", index=False)
        df_pendencias.to_excel(writer, sheet_name="Pendencias", index=False)

    # Formatação visual hierárquica, preservando as 17 colunas do modelo.
    # O estilo é aplicado principalmente à coluna de descrição para manter o arquivo leve.
    wb_out = load_workbook(caminho_saida)
    header_fill = PatternFill("solid", fgColor="17365D")
    header_font = Font(color="FFFFFF", bold=True)
    pai_fill = PatternFill("solid", fgColor="D9EAF7")
    receita_fill = PatternFill("solid", fgColor="F2F6FA")
    pai_font = Font(bold=True, color="17365D")
    receita_font = Font(bold=True)

    larguras_dados = {
        "A": 28, "B": 14, "C": 30, "D": 25, "E": 24, "F": 40,
        "G": 12, "H": 10, "I": 22, "J": 44, "K": 12, "L": 10,
        "M": 10, "N": 12, "O": 15, "P": 12, "Q": 18,
    }
    for ws in wb_out.worksheets:
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = f"A1:{get_column_letter(ws.max_column)}{ws.max_row}"
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        ws.row_dimensions[1].height = 32

    ws_dados = wb_out["Dados"]
    for col, width in larguras_dados.items():
        ws_dados.column_dimensions[col].width = width
    ws_pend = wb_out["Pendencias"]
    for col, width in {"A": 26, "B": 24, "C": 42, "D": 70, "E": 90}.items():
        ws_pend.column_dimensions[col].width = width
        ws_pend.column_dimensions[col].bestFit = False

    map_dados = {texto(cell.value): cell.column for cell in ws_dados[1]}
    idx_produto = map_dados.get("Produto e receita")
    idx_class = map_dados.get("Class")
    idx_cardapio = map_dados.get("Cardapio")
    for row_num, detalhe in enumerate(detalhes_visuais, start=2):
        nivel = min(int(detalhe["nivel"]), 15)
        if idx_produto:
            celula_produto = ws_dados.cell(row_num, idx_produto)
            celula_produto.alignment = Alignment(indent=nivel, vertical="center")
        classe_val = ws_dados.cell(row_num, idx_class).value if idx_class else None
        if detalhe["tipo"] == "pai":
            # Destaca a linha-pai nas colunas principais sem colorir as 17 células.
            if idx_produto:
                ws_dados.cell(row_num, idx_produto).fill = pai_fill
                ws_dados.cell(row_num, idx_produto).font = pai_font
            if idx_cardapio:
                ws_dados.cell(row_num, idx_cardapio).fill = pai_fill
                ws_dados.cell(row_num, idx_cardapio).font = pai_font
        elif classe_val in {"R", "S"}:
            if idx_produto:
                ws_dados.cell(row_num, idx_produto).fill = receita_fill
                ws_dados.cell(row_num, idx_produto).font = receita_font
            if idx_class:
                ws_dados.cell(row_num, idx_class).font = receita_font

    # Formatos numéricos alinhados ao modelo GRU JJ INT 0262:
    # Yield: percentual com 2 casas; Quantidade: até 12 casas decimais sem perder precisão;
    # Custo Unitario: exibido como um número normal com 2 casas decimais (ex.: #,##0.00)
    formatos_numericos = {
        "Yield": "0.00%",
        "Quantidade": "#,##0.############",
        "Custo Unitario": "#,##0.00",
    }
    for header, formato in formatos_numericos.items():
        col = map_dados.get(header)
        if col:
            for row_num in range(2, ws_dados.max_row + 1):
                ws_dados.cell(row_num, col).number_format = formato

    wb_out.save(caminho_saida)

    print(f"Arquivo gerado: {caminho_saida}")
    print(f"Linhas hierárquicas na aba Dados: {len(df_saida):,}".replace(",", "."))
    print(f"Pratos/cardápios incluídos: {sum(1 for d in detalhes_visuais if d['tipo'] == 'pai'):,}".replace(",", "."))
    print(f"Pendências para revisão: {len(df_pendencias):,}".replace(",", "."))
    print("A aba Dados contém pai, receitas filhas e ingredientes em ordem hierárquica; confira Pendencias.")
    return caminho_saida


# -----------------------------------------------------------------------------
# 5) EXECUÇÃO NO GOOGLE COLAB: SELEÇÃO DE ARQUIVOS E DOWNLOAD AUTOMÁTICO
# -----------------------------------------------------------------------------
def executar_no_colab() -> None:
    """Abre DUAS caixas de seleção separadas: primeiro LISTAGEM, depois RECEITAS."""
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError(
            "Esta função usa as caixas de upload do Google Colab. Em ambiente local, "
            "chame processar_arquivos(caminho_listagem, caminho_receitas)."
        ) from exc

    def selecionar_um_arquivo(mensagem: str, identificador: str) -> str:
        print("\n" + mensagem)
        enviados = files.upload()  # cada chamada abre uma caixa de seleção separada
        if not enviados:
            raise ValueError("Nenhum arquivo selecionado. Execute a célula novamente e escolha o arquivo pedido.")
        if len(enviados) != 1:
            raise ValueError(
                "Selecione apenas UM arquivo por caixa. Foram selecionados "
                f"{len(enviados)} arquivos. Execute novamente e selecione um de cada vez."
            )

        nome_original = next(iter(enviados.keys()))
        extensao = Path(nome_original).suffix.lower()
        if extensao not in {".xlsx", ".xlsm"}:
            raise ValueError(
                f"O arquivo '{nome_original}' não é .xlsx ou .xlsm. Converta-o para Excel e tente novamente."
            )

        # Faz uma cópia interna distinta para evitar sobrescrita caso os dois arquivos
        # tenham exatamente o mesmo nome original.
        nome_interno = f"__entrada_{identificador}{extensao}"
        if os.path.abspath(nome_original) != os.path.abspath(nome_interno):
            with open(nome_original, "rb") as origem, open(nome_interno, "wb") as destino:
                destino.write(origem.read())
        print(f"Arquivo selecionado para {identificador.upper()}: {nome_original}")
        return nome_interno

    # IMPORTANTE: são duas chamadas independentes a files.upload(), com caixas separadas.
    arq_listagem = selecionar_um_arquivo(
        "ETAPA 1 DE 2 — Selecione SOMENTE o arquivo LISTAGEM.xlsx.", "listagem"
    )
    arq_receitas = selecionar_um_arquivo(
        "ETAPA 2 DE 2 — Agora selecione SOMENTE o arquivo RECEITAS.xlsx.", "receitas"
    )

    saida = processar_arquivos(arq_listagem, arq_receitas)
    print(f"\nDownload do arquivo final: {saida}")
    from google.colab import files
    files.download(saida)


# No Colab, esta condição inicia o upload automaticamente ao executar a célula/script.
if __name__ == "__main__":
    executar_no_colab()



ETAPA 1 DE 2 — Selecione SOMENTE o arquivo LISTAGEM.xlsx.


Saving LISTEGEM_QANTAS SCL_MN 30_31.03.26_AGO26 - Copia.xlsx to LISTEGEM_QANTAS SCL_MN 30_31.03.26_AGO26 - Copia (3).xlsx
Arquivo selecionado para LISTAGEM: LISTEGEM_QANTAS SCL_MN 30_31.03.26_AGO26 - Copia (3).xlsx

ETAPA 2 DE 2 — Agora selecione SOMENTE o arquivo RECEITAS.xlsx.


Saving RECEITAS QANTAS SCL_MN 30_31.03.26_AGO26 - Copia.xlsx to RECEITAS QANTAS SCL_MN 30_31.03.26_AGO26 - Copia (3).xlsx
Arquivo selecionado para RECEITAS: RECEITAS QANTAS SCL_MN 30_31.03.26_AGO26 - Copia (3).xlsx
Arquivo gerado: MRP Costing Teknisa QF QANTAS AIRWAYS LIMITED.xlsx
Linhas hierárquicas na aba Dados: 8.133
Pratos/cardápios incluídos: 355
Pendências para revisão: 4
A aba Dados contém pai, receitas filhas e ingredientes em ordem hierárquica; confira Pendencias.

Download do arquivo final: MRP Costing Teknisa QF QANTAS AIRWAYS LIMITED.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>